# PISA → BigQuery: veri yükleme defteri
**CMEEP 2026 · Çalıştay 14 · Zekeriya Beşiroğlu**

Bu defter Drive'ınızdaki **PISA 2025** öğrenci ve okul dosyalarını bulur, gereken sütunları seçer
ve BigQuery'de `pisa.ogrenci` ve `pisa.okul` tablolarına yükler. **Demodan önce bir kez çalıştırın**
(toplam ~20–30 dk); demoda yalnızca BigQuery konsolunda SQL çalıştırılır.

Çalıştırma: 1. hücreye proje kimliğinizi yazın, sonra **Çalışma zamanı → Tümünü çalıştır**.
Google hesabınızla iki kez izin vermeniz istenir (BigQuery ve Drive).

## 1 · Ayarlar
Yalnızca `PROJE_ID` satırını değiştirmeniz yeterli.

In [ ]:
PROJE_ID = "proje-kimliginiz"     # ← Google Cloud proje kimliği (BigQuery konsolunda sol üstte)
VERI_SETI = "pisa"
KONUM = "EU"                      # veri seti konumu; sonradan değiştirilemez

# "otomatik": Drive'ım'da adında STU / SCH geçen SPSS dosyaları aranır ve seçilir.
# Yanlış dosya seçilirse buraya TAM yolu yazın, örn.
#   DONGULER = {2025: "/content/drive/MyDrive/PISA2025/CY09MSP_STU_QQQ.zip"}
DONGULER = {2025: "otomatik"}         # ÖĞRENCİ dosyası (STU) — zorunlu
OKUL_DOSYALARI = {2025: "otomatik"}   # OKUL dosyası (SCH) — 13 numaralı SQL için
# Öğretmen (TCH) ve bilişsel madde (COG) dosyaları bu demoda kullanılmaz.

## 2 · Kurulum, Google girişi ve Drive

In [ ]:
!pip install -q pyreadstat google-cloud-bigquery db-dtypes pyarrow
from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")
print("Giriş tamam.")

In [ ]:
# Drive'da PISA dosyalarını bulur ve "otomatik" olanları doldurur
import os
KOK = "/content/drive/MyDrive"
print("Drive'ım bağlı mı:", os.path.isdir(KOK))
bulunan = []
for kok, klasorler, dosyalar in os.walk(KOK):
    if kok.count(os.sep) - KOK.count(os.sep) >= 5:
        klasorler[:] = []
    for d in dosyalar:
        if d.upper().endswith((".ZIP", ".SAV")):
            bulunan.append(os.path.join(kok, d))


def sec(anahtar):
    """Adında anahtar (STU/SCH) geçen, QQQ (anket) dosyasını tercih eden, SAS/COG/TIM olmayan dosya."""
    adaylar = []
    for yol in bulunan:
        ad = os.path.basename(yol).upper()
        if anahtar in ad and not any(k in ad for k in ("SAS", "COG", "TIM", "TCH")):
            adaylar.append(yol)
    adaylar.sort(key=lambda y: ("QQQ" not in os.path.basename(y).upper(), -os.path.getsize(y)))
    return adaylar[0] if adaylar else None


print("Drive'daki .zip / .sav dosyaları:")
for yol in sorted(bulunan):
    print(f"  {os.path.getsize(yol) / 1e9:6.2f} GB  {yol}")
for sozluk_, anahtar, ad in [(DONGULER, "STU", "Öğrenci"), (OKUL_DOSYALARI, "SCH", "Okul")]:
    for dongu, kaynak in list(sozluk_.items()):
        if kaynak == "otomatik":
            sozluk_[dongu] = sec(anahtar)
        print(f"→ {ad} {dongu}: {sozluk_[dongu]}")
if not DONGULER.get(2025):
    raise FileNotFoundError("Öğrenci dosyası bulunamadı. Drive'a yüklendiğinden emin olun "
                            "ya da 1. hücreye tam yolu yazın.")
OKUL_DOSYALARI = {d: k for d, k in OKUL_DOSYALARI.items() if k}   # okul dosyası yoksa atla

## 3 · Yardımcı fonksiyonlar
Değişken listesi burada; eklemek istediğiniz anket endekslerini `ANKET` listesine yazın.

In [ ]:
import glob, os, zipfile, urllib.request
import pandas as pd
import pyreadstat

# Analizde kullanılacak değişkenler. Dosyada olmayanlar atlanır ve listelenir.
KIMLIK = ["CNT", "CNTRYID", "CNTSCHID", "CNTSTUID", "OECD", "STRATUM", "PROGN"]
ANKET = ["ST004D01T", "ESCS", "HOMEPOS", "BELONG", "ANXMAT", "MATHEFF",
         "REPEAT", "ICTRES", "DISCLIM", "TEACHSUP", "FAMSUP", "EXPOFA"]
OLASI_DEGERLER = [f"PV{i}{alan}" for alan in ("MATH", "READ", "SCIE") for i in range(1, 11)]
AGIRLIKLAR = ["W_FSTUWT"] + [f"W_FSTURWT{i}" for i in range(1, 81)]
ISTENEN = KIMLIK + ANKET + OLASI_DEGERLER + AGIRLIKLAR
ZORUNLU = ["CNT", "CNTSTUID", "W_FSTUWT", "PV1MATH", "W_FSTURWT80"]
ETIKETLE = ["STRATUM", "PROGN"]        # değer etiketleri ayrı sütuna yazılır (…_ETIKET)

# Okul anketi: yerleşim yeri büyüklüğü, okul türü, okul büyüklüğü
OKUL_ISTENEN = ["CNT", "CNTSCHID", "SC001Q01TA", "SCHLTYPE", "SCHSIZE"]
OKUL_ZORUNLU = ["CNT", "CNTSCHID"]
OKUL_ETIKETLE = ["SC001Q01TA", "SCHLTYPE"]


def hazirla(kaynak, klasor):
    """Kaynak bir web adresi, .zip ya da .sav olabilir. SPSS (.sav) dosyasının yolunu döndürür.
    Her kaynak tek bir dosya türü (yalnızca öğrenci ya da yalnızca okul) içermelidir."""
    if kaynak.lower().endswith(".sav"):
        if not os.path.exists(kaynak):
            raise FileNotFoundError(f"Dosya yok: {kaynak} (Drive bağlandı mı, ad doğru mu?)")
        sav = kaynak
    else:
        os.makedirs(klasor, exist_ok=True)
        if kaynak.startswith("http"):
            zip_yolu = os.path.join(klasor, os.path.basename(kaynak))
            if not os.path.exists(zip_yolu):
                print("İndiriliyor:", kaynak, "(birkaç dakika sürebilir)")
                urllib.request.urlretrieve(kaynak, zip_yolu)
        else:
            zip_yolu = kaynak
            if not os.path.exists(zip_yolu):
                raise FileNotFoundError(f"Dosya yok: {zip_yolu} (Drive bağlandı mı, ad doğru mu?)")
        print("Açılıyor:", zip_yolu)
        with zipfile.ZipFile(zip_yolu) as z:
            z.extractall(klasor)
        savlar = glob.glob(os.path.join(klasor, "**", "*.[sS][aA][vV]"), recursive=True)
        if not savlar:
            raise FileNotFoundError(f"{zip_yolu} içinde .sav dosyası bulunamadı. "
                                    "SAS değil SPSS sürümünü (…_SPSS.zip) indirin.")
        sav = max(savlar, key=os.path.getsize)
    print("SPSS dosyası:", sav, f"({os.path.getsize(sav) / 1e9:.2f} GB)")
    return sav


indir_ve_ac = hazirla      # eski ad da çalışsın


def degisken_ara(sav_yolu, *kelimeler):
    """Değişken adında ya da etiketinde kelimelerden biri geçenleri listeler (yalnızca üst veri; hızlı)."""
    _, meta = pyreadstat.read_sav(sav_yolu, metadataonly=True)
    satirlar = [(ad, meta.column_names_to_labels.get(ad) or "") for ad in meta.column_names]
    kelimeler = [k.lower() for k in kelimeler]
    bulunan = [(a, e) for a, e in satirlar if any(k in a.lower() or k in e.lower() for k in kelimeler)]
    return pd.DataFrame(bulunan, columns=["degisken", "etiket"])


def degisken_kontrol(sav_yolu, istenen=None):
    """Okumadan önce: hangi değişkenler dosyada var, hangileri yok?"""
    istenen = istenen or ISTENEN
    _, meta = pyreadstat.read_sav(sav_yolu, metadataonly=True)
    mevcut = set(meta.column_names)
    eksik = [d for d in istenen if d not in mevcut]
    print(f"{os.path.basename(sav_yolu)}: {meta.number_rows or '?'} satır, {len(mevcut)} değişken")
    print(f"  İstenen {len(istenen)} değişkenden {len(istenen) - len(eksik)} tanesi var.")
    if eksik:
        print("  EKSİK:", ", ".join(eksik))
        print("  → degisken_ara(yol, 'anxiety', 'belong' …) ile karşılığını bulup listeye ekleyin.")
    return eksik


def oku(sav_yolu, dongu, istenen=None, zorunlu=None, etiketle=None, islemci=2):
    """Yalnızca gereken sütunları okur; SPSS kayıp kodlarını (95, 97, 98, 99…) NaN yapar."""
    istenen = istenen or ISTENEN
    zorunlu = zorunlu or ZORUNLU
    etiketle = ETIKETLE if etiketle is None else etiketle
    _, meta = pyreadstat.read_sav(sav_yolu, metadataonly=True)
    mevcut = set(meta.column_names)
    eksik_zorunlu = [d for d in zorunlu if d not in mevcut]
    if eksik_zorunlu:
        raise KeyError(f"{dongu} dosyasında zorunlu değişkenler yok: {eksik_zorunlu}")
    secilen = [d for d in istenen if d in mevcut]
    atlanan = [d for d in istenen if d not in mevcut]
    print(f"{dongu}: {len(secilen)} değişken okunacak ({meta.number_rows or '?'} satır).")
    if atlanan:
        print(f"  Dosyada olmadığı için atlanan {len(atlanan)} değişken:", ", ".join(atlanan))

    # user_missing=False (varsayılan): kullanıcı tanımlı kayıp değerler NaN olur.
    df, meta = pyreadstat.read_file_multiprocessing(
        pyreadstat.read_sav, sav_yolu, num_processes=islemci, usecols=secilen)
    df.insert(0, "CYCLE", int(dongu))
    df["CNT"] = df["CNT"].astype(str).str.strip()
    for d in etiketle:                       # ör. STRATUM → STRATUM_ETIKET
        etiketler = meta.variable_value_labels.get(d)
        if d in df.columns and etiketler:
            df[d + "_ETIKET"] = df[d].map(etiketler)
            print(f"  {d}_ETIKET eklendi ({len(etiketler)} etiket)")
    for d in df.columns:                     # karışık tipli metin sütunlarını düzelt
        if df[d].dtype == object:
            df[d] = df[d].astype("string")
    sozluk = pd.DataFrame({
        "CYCLE": int(dongu),
        "degisken": meta.column_names,
        "etiket": [meta.column_names_to_labels.get(c) for c in meta.column_names],
    })
    return df, sozluk


def agirlikli_ortalama(df, ulke, alan="MATH"):
    """Hızlı kontrol: bir ülkenin 10 PV ile ağırlıklı ortalaması (pandas ile)."""
    d = df[df["CNT"] == ulke]
    w = d["W_FSTUWT"]
    return sum((d[f"PV{i}{alan}"] * w).sum() / w.sum() for i in range(1, 11)) / 10


## 4 · Değişken kontrolü (hızlı, ~1 dk)
Tam okumadan önce her dosyada hangi değişkenlerin olduğuna bakar. **2025'te adı değişen değişkenler**
burada "EKSİK" olarak görünür. Karşılığını bulmak için örnek:
`degisken_ara(SAVLAR[2025], "anxiety", "belong", "self-efficacy")` → bulduğunuz adı 3. hücredeki
`ANKET` listesine ekleyip 3. hücreyi ve bu hücreyi yeniden çalıştırın.

In [ ]:
SAVLAR, OKUL_SAVLAR = {}, {}
for dongu, kaynak in DONGULER.items():
    SAVLAR[dongu] = hazirla(kaynak, f"/content/pisa{dongu}_ogrenci")
    degisken_kontrol(SAVLAR[dongu])
for dongu, kaynak in OKUL_DOSYALARI.items():
    OKUL_SAVLAR[dongu] = hazirla(kaynak, f"/content/pisa{dongu}_okul")
    degisken_kontrol(OKUL_SAVLAR[dongu], OKUL_ISTENEN)

In [ ]:
# İsteğe bağlı: 2025 dosyasında değişken arama
degisken_ara(SAVLAR[max(SAVLAR)], "anxiety", "belong", "efficacy", "gender", "programme", "stratum")

## 5 · Oku ve birleştir
Her döngü için ~10–15 dk sürebilir. Bellek hatası alırsanız `ANKET` listesini kısaltın.

In [ ]:
parcalar, sozlukler = [], []
for dongu, sav in SAVLAR.items():
    df, sozluk = oku(sav, dongu)
    parcalar.append(df)
    sozlukler.append(sozluk)
    print(f"  {dongu}: {len(df):,} öğrenci, {df['CNT'].nunique()} ülke")
    print(f"  Hızlı kontrol · TUR {dongu} matematik ≈ {agirlikli_ortalama(df, 'TUR'):.1f}")

okul_parcalari = []
for dongu, sav in OKUL_SAVLAR.items():
    okul_df, okul_sozluk = oku(sav, dongu, istenen=OKUL_ISTENEN,
                               zorunlu=OKUL_ZORUNLU, etiketle=OKUL_ETIKETLE)
    okul_parcalari.append(okul_df)
    sozlukler.append(okul_sozluk)
    print(f"  {dongu}: {len(okul_df):,} okul")

ogrenci = pd.concat(parcalar, ignore_index=True, sort=False)
sozluk = pd.concat(sozlukler, ignore_index=True)
ogrenci.to_parquet("/content/ogrenci.parquet", index=False)
print(f"Öğrenci: {len(ogrenci):,} satır × {ogrenci.shape[1]} sütun")
if okul_parcalari:
    okul = pd.concat(okul_parcalari, ignore_index=True, sort=False)
    okul.to_parquet("/content/okul.parquet", index=False)
    print(f"Okul: {len(okul):,} satır")

## 6 · BigQuery'ye yükle
Tablo `CYCLE, CNT` ile kümelenir: yıl ve ülke filtreleri daha az veri tarar.

In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project=PROJE_ID)

veri_seti = bigquery.Dataset(f"{PROJE_ID}.{VERI_SETI}")
veri_seti.location = KONUM
client.create_dataset(veri_seti, exists_ok=True)

ayar = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition="WRITE_TRUNCATE",
    clustering_fields=["CYCLE", "CNT"],
)
for tablo in ["ogrenci", "okul"]:
    yol = f"/content/{tablo}.parquet"
    if not os.path.exists(yol):
        continue
    with open(yol, "rb") as f:
        client.load_table_from_file(f, f"{PROJE_ID}.{VERI_SETI}.{tablo}", job_config=ayar).result()
    print(f"{tablo} tablosu:", client.get_table(f"{PROJE_ID}.{VERI_SETI}.{tablo}").num_rows, "satır")

client.load_table_from_dataframe(
    sozluk, f"{PROJE_ID}.{VERI_SETI}.degisken_sozluk",
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
).result()
print("degisken_sozluk tablosu yüklendi.")

## 7 · BigQuery'de doğrulama
Beklenen (MEB, PISA 2025 Türkiye raporu): **TUR 2025 matematik → 462**. Tutmuyorsa ağırlık ya da PV adlarını kontrol edin.

In [ ]:
sorgu = f"""
WITH uzun AS (
  SELECT CYCLE, W_FSTUWT AS w, puan
  FROM `{PROJE_ID}.{VERI_SETI}.ogrenci`
  UNPIVOT (puan FOR pv IN (PV1MATH, PV2MATH, PV3MATH, PV4MATH, PV5MATH,
                           PV6MATH, PV7MATH, PV8MATH, PV9MATH, PV10MATH))
  WHERE CNT = 'TUR'
)
SELECT CYCLE, ROUND(SUM(w * puan) / SUM(w), 1) AS matematik
FROM uzun GROUP BY CYCLE ORDER BY CYCLE
"""
client.query(sorgu).to_dataframe()

## Sonraki adım
BigQuery konsolunu açın (console.cloud.google.com/bigquery) ve `sql/` klasöründeki
dosyaları sırayla çalıştırın. **06** ve **08** dosyalarındaki `CREATE MODEL` bölümlerini
demodan önce çalıştırın.